# 01 Data cleaning (survey 2025-2026)

Input: the survey export with GPS already removed (`data/raw/`, not in the repo).
Output: `data/processed/2025-2026_survey_clean.csv`, used by notebooks 02 and 03.

Steps
1. Keep only the coded answer columns (open-text answers and stray columns are dropped).
2. Check that no personal-data column is left.
3. Convert the counts to numbers.
4. Apply the exclusion rules, one at a time, and count what each rule removes.
5. Compute deaths and the loss rate per apiary.
6. Compare `in - uit` with the reported number of dead colonies.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

# Run from the repo root or from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_FILE = ROOT / "data" / "raw" / "2026_surveydata_cleaned_exclude_gps.csv"
CLEAN_FILE = ROOT / "data" / "processed" / "2025-2026_survey_clean.csv"
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

HIVES_IN, HIVES_OUT, HIVES_DEAD = "2025_winter_in", "2026_winter_uit", "2025_winter_dood"


## 1. Load and keep only the coded columns

In [2]:
KEEP = [
    "id_location", "id_beekeeper", HIVES_IN, HIVES_OUT, HIVES_DEAD,
    "imker_methode", "winter_food", "varroa_treatment",
    "varroa_observation", "aziatische_hoornaar", "deformed_wings",
    "province", "bijenstanden_count",
    "honingdracht_voorjaar", "honingdracht_zomer", "honingdracht_nazomer",
    "stuifmeel_voorjaar", "stuifmeel_zomer", "stuifmeel_nazomer",
]

raw = pd.read_csv(RAW_FILE, encoding="utf-8-sig", low_memory=False)
print(f"Raw file: {raw.shape[0]} rows, {raw.shape[1]} columns")

df = raw[[c for c in KEEP if c in raw.columns]].copy()
print(f"Kept {df.shape[1]} coded columns, dropped {raw.shape[1] - df.shape[1]}")

Raw file: 833 rows, 103 columns
Kept 19 coded columns, dropped 84


## 2. Privacy check
Stops the notebook if any column name looks like personal data or free text.

In [3]:
PII_PATTERNS = [r"e-?mail", r"naam", r"\bname\b", r"gps", r"\blat", r"\blon", r"coord",
                r"postcode", r"adres", r"address", r"telefoon", r"phone", r"_origin$",
                r"opmerking", r"comment", r"ziektes"]
pii = [c for c in df.columns if any(re.search(p, c, re.IGNORECASE) for p in PII_PATTERNS)]
assert not pii, f"Personal or free-text columns left: {pii}"
print("No personal-data columns")

No personal-data columns


## 3. Convert to numbers
The number of dead colonies was typed freely (`Nee`, `o`, `3!`, `januari 1 volk`), so it gets its own parser.

In [4]:
def parse_count(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower()
    if text in {"nee", "no", "n", "o", "geen"}:
        return 0.0
    match = re.search(r"\d+(?:[,.]\d+)?", text)
    return float(match.group(0).replace(",", ".")) if match else np.nan


numeric_cols = [c for c in KEEP if c in df.columns and c not in (HIVES_DEAD, "province")]
for col in numeric_cols:
    before = df[col].notna().sum()
    df[col] = pd.to_numeric(df[col], errors="coerce")
    lost = before - df[col].notna().sum()
    if lost:
        print(f"{col}: {lost} values could not be read as numbers")
df[HIVES_DEAD] = df[HIVES_DEAD].map(parse_count)

## 4. Exclusion rules

In [5]:
rules = [
    ("no apiary id (summary rows)",  lambda d: d["id_location"].isna()),
    ("hive counts missing",          lambda d: d[HIVES_IN].isna() | d[HIVES_OUT].isna()),
    ("0 hives going into winter",    lambda d: d[HIVES_IN] <= 0),
    ("more hives out than in",       lambda d: d[HIVES_OUT] > d[HIVES_IN]),
]

log = [{"step": "loaded", "dropped": 0, "remaining": len(df)}]
for name, rule in rules:
    drop = rule(df).fillna(False)
    df = df.loc[~drop]
    log.append({"step": name, "dropped": int(drop.sum()), "remaining": len(df)})

log = pd.DataFrame(log)
log.to_csv(RESULTS / "01_cleaning_log.csv", index=False)
log

,step,dropped,remaining
0,loaded,0,833
1,no apiary id (summary rows),1,832
2,hive counts missing,0,832
3,0 hives going into winter,4,828
4,more hives out than in,0,828


## 5. Deaths and loss rate
`deaths = in - uit` and `mortality = deaths / in`, per apiary. Group loss rates later are always
hive-weighted: total deaths / total hives in.

In [6]:
df = df.reset_index(drop=True)
df["deaths"] = df[HIVES_IN] - df[HIVES_OUT]
df["mortality"] = df["deaths"] / df[HIVES_IN]

total_in, total_dead = df[HIVES_IN].sum(), df["deaths"].sum()
print(f"Apiaries: {len(df)}   Beekeepers: {df['id_beekeeper'].nunique()}")
print(f"Hives in: {total_in:.0f}   Dead: {total_dead:.0f}   Loss: {100 * total_dead / total_in:.2f}%")

Apiaries: 828   Beekeepers: 692
Hives in: 4843   Dead: 1047   Loss: 21.62%


## 6. Check: `in - uit` vs reported dead colonies
They differ when colonies were merged, sold, split or bought over winter. The analysis uses `in - uit`.

In [7]:
check = df[["id_location", HIVES_IN, HIVES_OUT, "deaths", HIVES_DEAD]].copy()
check["difference"] = check["deaths"] - check[HIVES_DEAD]
mismatch = check[check[HIVES_DEAD].notna() & check["difference"].ne(0)]
mismatch.to_csv(RESULTS / "01_death_definition_mismatches.csv", index=False)
print(f"{len(mismatch)} of {len(df)} apiaries differ")
mismatch["difference"].value_counts().sort_index().rename("apiaries").to_frame()

170 of 828 apiaries differ


,apiaries
difference,
-28.0,1
-13.0,1
-4.0,2
-2.0,1
-1.0,6
1.0,90
2.0,35
3.0,16
4.0,7


## 7. Missing answers per variable

In [8]:
variables = ["imker_methode", "winter_food", "varroa_treatment",
             "varroa_observation", "aziatische_hoornaar", "deformed_wings"]
valid_n = pd.DataFrame({
    "apiaries_answered": [int(df[v].notna().sum()) for v in variables],
    "apiaries_missing": [int(df[v].isna().sum()) for v in variables],
}, index=variables)
valid_n.to_csv(RESULTS / "01_valid_n_per_variable.csv")
valid_n

,apiaries_answered,apiaries_missing
imker_methode,827,1
winter_food,826,2
varroa_treatment,828,0
varroa_observation,828,0
aziatische_hoornaar,828,0
deformed_wings,828,0


## 8. Save

In [9]:
CLEAN_FILE.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEAN_FILE, index=False)
print(f"Saved {df.shape[0]} rows x {df.shape[1]} columns to data/processed/{CLEAN_FILE.name}")

Saved 828 rows x 21 columns to data/processed/2025-2026_survey_clean.csv
